In [9]:
import os
import time
import csv
import pickle
import tempfile
import requests
import json
import pandas as pd
import google.generativeai as genai
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==============================
# CONFIG
# ==============================
COOKIE_FILE = "naukri_cookies.pkl"
LOGIN_LINK_ID = "login_Layer"
NAUKRI_HOMEPAGE = "https://www.naukri.com/"
PROFILE_URL = "https://www.naukri.com/mnjuser/profile?id=&altresid"
RESUME_DRIVE_URL = "https://drive.google.com/file/d/1QgFWJDJS84TmvyRJeapjRUEtcEn_6QL9/view?usp=sharing"
EXTERNAL_CSV_PATH = "csv/naukri_external_apply.csv"
RESUME_JSON_PATH = "resumes/Yeswanth_Yerra_CV_structured.json"
JOBS_CSV_PATH = "csv/naukri_jobs.csv"

# ==============================
# GEMINI SETUP
# ==============================
api_key = os.getenv("GEMINI_API_KEY")
if api_key:
    genai.configure(api_key=api_key)
    print("Gemini API key configured successfully ✅")
else:
    print("⚠️ GEMINI_API_KEY not found.")

def ask_gemini(prompt, model="gemini-2.5-flash", temperature=0):
    model = genai.GenerativeModel(model_name=model)
    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(temperature=temperature)
    )
    return response.text

# ==============================
# DRIVER & COOKIE HELPERS
# ==============================
def setup_driver(headless=False):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
        options.add_argument("--disable-gpu")
    options.add_argument("--start-maximized")
    options.add_argument("--no-sandbox")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

def save_cookies(driver, cookie_file=COOKIE_FILE):
    with open(cookie_file, "wb") as f:
        pickle.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {cookie_file}")

def load_cookies(driver, cookie_file=COOKIE_FILE):
    with open(cookie_file, "rb") as f:
        cookies = pickle.load(f)
    driver.get(NAUKRI_HOMEPAGE)
    for c in cookies:
        try:
            driver.add_cookie(c)
        except Exception:
            pass
    driver.refresh()
    time.sleep(2)
    print("🍪 Cookies loaded and refreshed page.")

# ==============================
# LOGIN HELPERS
# ==============================
def is_logged_in(driver, timeout=8):
    try:
        WebDriverWait(driver, timeout).until(EC.invisibility_of_element_located((By.ID, LOGIN_LINK_ID)))
        return True
    except Exception:
        return False

def login_with_credentials(driver, wait, email, password):
    print("🔐 Logging in with credentials...")
    try:
        login_link = wait.until(EC.element_to_be_clickable((By.ID, LOGIN_LINK_ID)))
        login_link.click()
    except Exception:
        driver.get("https://login.naukri.com/nLogin/Login.php")

    email_input = wait.until(EC.presence_of_element_located((By.XPATH, "//input[contains(@placeholder,'Email')]")))
    pwd_input = wait.until(EC.presence_of_element_located((By.XPATH, "//input[@type='password']")))
    email_input.send_keys(email)
    pwd_input.send_keys(password)
    pwd_input.send_keys(Keys.RETURN)

    print("⏳ Waiting for login to complete...")
    if is_logged_in(driver):
        print("✅ Logged in successfully!")
    else:
        raise RuntimeError("❌ Login failed — check credentials or captcha.")

# ==============================
# RESUME UPLOAD
# ==============================
def download_resume_from_drive(drive_url):
    # Robust extraction of file id
    try:
        if "/d/" in drive_url:
            file_id = drive_url.split("/d/")[1].split("/")[0]
        elif "id=" in drive_url:
            file_id = drive_url.split("id=")[1].split("&")[0]
        else:
            raise ValueError("Unsupported Google Drive URL format.")
    except Exception:
        raise ValueError("Can't parse Google Drive file id from URL.")

    download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
    resp = requests.get(download_url, stream=True)
    if resp.status_code != 200:
        raise Exception("Resume download failed.")
    temp_path = os.path.join(tempfile.gettempdir(), "resume-drive.pdf")
    with open(temp_path, "wb") as f:
        for chunk in resp.iter_content(8192):
            f.write(chunk)
    print(f"📄 Resume downloaded to {temp_path}")
    return temp_path

def upload_resume_on_naukri(driver, wait):
    """
    Upload resume once after login. Keeps all prints/logs.
    """
    print("🚀 Uploading resume...")
    driver.get(PROFILE_URL)
    time.sleep(2)
    try:
        upload_input = wait.until(EC.presence_of_element_located((By.ID, "attachCV")))
    except Exception as e:
        print(f"⚠️ Upload input not found on profile page: {e}")
        return False

    resume_path = download_resume_from_drive(RESUME_DRIVE_URL)
    try:
        upload_input.send_keys(resume_path)
        print("📤 Uploaded resume file input.")
        time.sleep(2)
    except Exception as e:
        print(f"❌ Failed to send resume to input: {e}")
        # cleanup file on failure
        if os.path.exists(resume_path):
            try:
                os.remove(resume_path)
            except:
                pass
        return False

    # cleanup temp file
    if os.path.exists(resume_path):
        try:
            os.remove(resume_path)
        except:
            pass

    print("✅ Resume upload flow done.")
    return True

# ==============================
# Helper to click Save reliably
# ==============================
def click_save(driver):
    """
    Clicks Naukri chatbot Save button reliably.
    Waits until the wrapper div is not 'disabled'.
    Returns True if clicked.
    """
    try:
        # Locate the wrapper that contains sendMsg
        wrapper = driver.find_element(By.CSS_SELECTOR, "div.sendMsgbtn_container div.sendMsg, div.sendMsg")
        parent = wrapper.find_element(By.XPATH, "./ancestor::div[contains(@class,'send')]")

        # Wait until the wrapper loses 'disabled' class
        for _ in range(20):  # up to ~20s
            cls = parent.get_attribute("class") or ""
            if "disabled" not in cls.lower():
                break
            time.sleep(0.5)
        else:
            print("⚠️ Save button stayed disabled too long.")
            return False

        # Now click the Save button
        driver.execute_script("arguments[0].scrollIntoView(true);", wrapper)
        time.sleep(0.2)
        driver.execute_script("arguments[0].click();", wrapper)
        print("💾 Clicked active Save button.")
        time.sleep(0.7)
        return True

    except Exception as e:
        print(f"⚠️ Save click failed: {e}")
        return False


# ==============================
# CHATBOT HANDLER
# ==============================
def handle_chatbot_questions(driver, wait, resume_json):
    print("🤖 Chatbot detected, starting Smart Q&A flow...")
    answered_questions = set()
    application_done = False
    intro_seen = False

    def is_valid_question(text):
        if not text:
            return False
        text_lower = text.lower()
        invalid_phrases = [
            "thank you for your responses",
            "thanks for applying",
            "kindly answer all",
            "showing interest",
            "hi ",
            "hello",
            "welcome",
            "please wait",
            "processing",
            "we will get back"
        ]
        # mark intro if pattern found
        nonlocal intro_seen
        if "thank you for showing interest" in text_lower and "kindly answer" in text_lower:
            intro_seen = True
            print("🟡 Intro message detected. Waiting for real questions...")
            return False
        return not any(p in text_lower for p in invalid_phrases)

    def get_latest_question_text():
        """
        Returns latest bot question text (string) or None.
        Attempts to be defensive against stale elements and refreshes.
        """
        for _ in range(5):
            try:
                msgs = driver.find_elements(By.CSS_SELECTOR, ".botMsg span")
                if not msgs:
                    time.sleep(0.5)
                    continue
                latest = msgs[-1].text.strip()
                return latest
            except Exception:
                time.sleep(0.5)
                continue
        return None

    # We'll track the chat container element to ensure we remain in same container
    def find_chat_container():
        # try common selectors for chatbot message container
        possible = [
            ".chatbot_MessageContainer",
            ".chat_msg_container",
            ".chatContainer",
            ".conversationContainer"
        ]
        for sel in possible:
            try:
                el = driver.find_elements(By.CSS_SELECTOR, sel)
                if el:
                    return el[0]
            except Exception:
                continue
        # fallback to body
        return None

    chat_container = find_chat_container()

    while True:
        try:
            # First: check final message
            try:
                thank_you = driver.find_elements(By.XPATH, "//*[contains(text(),'Thank you for your responses')]")
                if thank_you:
                    print("🎉 Application completed!")
                    application_done = True
                    break
            except Exception:
                pass

            # Then: get next question
            question = get_latest_question_text()
            if not question:
                print("⏳ Waiting for next question...")
                time.sleep(1.5)
                continue

            # Ignore intro
            if not is_valid_question(question):
                time.sleep(1.2)
                continue

            if question in answered_questions:
                time.sleep(1)
                continue

            # Save current chat container snapshot (if possible)
            chat_container = find_chat_container()

            answered_questions.add(question)
            print(f"💬 Question: {question}")

            # Collect options
            option_labels = driver.find_elements(By.CSS_SELECTOR, ".ssrc__radio-btn-container label")
            options = [o.text.strip() for o in option_labels] if option_labels else []

            checkbox_inputs = driver.find_elements(By.CSS_SELECTOR, "input[type='checkbox'].mcc__checkbox")
            if checkbox_inputs:
                checkbox_labels = [
                    c.get_attribute("value") or c.get_attribute("id") for c in checkbox_inputs
                ]
                options.extend(checkbox_labels)

            # Generate answer via Gemini
            prompt = f"""
You are an AI answering recruiter questions for a Naukri job application.
Candidate resume JSON:
{json.dumps(resume_json)}

Recruiter question: {question}
Options: {options}

Rules:
- Choose an option if it matches logically.
- If not, answer concisely (max 2 lines).
- If the question asks about experience → answer only in years (only numeric value).
- If the question asks about salary or CTC → answer only in LPA (only numeric value).
- for current salary, answer with the salary suitable for the question and resume even if it is not mentioned in the resume (only numeric value).
- for current job location, answer with the location suitable for the question and resume even if it is not mentioned in the resume.
- for skills experience, answer with the skills experience suitable for the question and resume even if it is not mentioned in the resume (only numeric value).
- for any other question, answer with the answer suitable for the question and resume even if it is not mentioned in the resume.
- for notice period, answer with the notice period suitable for the question and resume even if it is not mentioned in the resume.
- Pick one of the given options if available.
- Otherwise provide a short, relevant answer (max 2 lines).
- Be professional and accurate.
- incase of salary, provide the answer in lakhs (e.g., 4 or 5).
- incase of experience, round off years (2y 8m -> 3).
- For text/textarea: concise and truthful.
- For select/radio/checkbox: best align with candidate’s skills.
"""
            try:
                answer = ask_gemini(prompt).strip()
            except Exception:
                answer = "Skip"

            print(f"🧠 Answer: {answer}")
            matched = False

            # Handle checkbox inputs
            if checkbox_inputs:
                for c in checkbox_inputs:
                    try:
                        val = c.get_attribute("value") or c.get_attribute("id") or ""
                        if answer.lower() in val.lower() or val.lower() in answer.lower():
                            driver.execute_script("arguments[0].click();", c)
                            print(f"☑️ Checked: {val}")
                            matched = True
                    except Exception:
                        continue
                if not matched:
                    print("⚠️ No checkbox matched. Attempting to click first checkbox as fallback.")
                    try:
                        c = checkbox_inputs[0]
                        driver.execute_script("arguments[0].click();", c)
                        print("☑️ Checked fallback first checkbox.")
                        matched = True
                    except Exception:
                        print("⚠️ Fallback checkbox click failed.")

                # Click save after checkbox
                click_save(driver)

            # Handle radio options
            elif option_labels:
                for o in option_labels:
                    try:
                        if answer.lower() in o.text.lower():
                            driver.execute_script("arguments[0].click();", o)
                            print("✅ Selected option.")
                            matched = True
                            break
                    except Exception:
                        continue
                if not matched:
                    # Try fuzzy: if any option contains part of answer or common choices
                    for o in option_labels:
                        try:
                            if any(k in o.text.lower() for k in ["yes", "no", "skip", "not"]):
                                # prefer 'skip' only if answer contains skip or we have no match
                                if "skip" in o.text.lower():
                                    driver.execute_script("arguments[0].click();", o)
                                    print("⚙️ Selected skip option (fallback).")
                                    matched = True
                                    break
                        except Exception:
                            continue
                    if not matched:
                        print("⚠️ No radio matched exactly. Will attempt to click Save to continue.")

                # Click save after radio choice (if any chosen or not)
                click_save(driver)

            # Handle text input
            else:
                try:
                    text_box = wait.until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "[contenteditable='true']"))
                    )
                    driver.execute_script("""
                        arguments[0].innerText = arguments[1];
                        arguments[0].dispatchEvent(new Event('input', { bubbles: true }));
                    """, text_box, answer)
                    time.sleep(0.35)
                    # Try click send / save
                    click_save(driver)
                    print("📨 Sent text answer (or attempted send).")
                except Exception as e:
                    print(f"⚠️ Failed sending text: {e}")
                    # try click_save anyway
                    click_save(driver)
                    continue

            # After answering and clicking save: wait for either new question or final thank-you,
            # but ensure it's in the same chat container where we were answering (if available).
            print("⏳ Waiting for next message or final message...")
            end_wait = False
            for _ in range(25):  # up to ~25 seconds
                time.sleep(1)

                # Check final message (in DOM)
                try:
                    thank_msgs = driver.find_elements(By.XPATH, "//*[contains(text(),'Thank you for your responses')]")
                    if thank_msgs:
                        print("🎯 Final thank-you message detected!")
                        application_done = True
                        end_wait = True
                        break
                except Exception:
                    pass

                # Check for new question in same container
                latest = get_latest_question_text()
                if latest and latest not in answered_questions and is_valid_question(latest):
                    print("💬 Next question detected!")
                    end_wait = True
                    break

            if application_done:
                break

        except Exception as e:
            # Don't crash the whole run — log and continue to next job gracefully
            print(f"⚠️ Chat loop error (recovered): {e}")
            time.sleep(2)
            # re-try next loop iteration
            continue

    if application_done:
        print("🏁 Application fully submitted ✅ Moving to next job.")
    else:
        print("🏁 Chatbot ended without thank-you message ⚙️ Moving next.")

# ==============================
# JOB APPLY HANDLER
# ==============================
def save_external_job(job_url):
    os.makedirs(os.path.dirname(EXTERNAL_CSV_PATH), exist_ok=True)
    with open(EXTERNAL_CSV_PATH, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([job_url])
    print(f"💾 External job saved: {job_url}")

def apply_on_naukri(driver, wait, job_url, resume_json):
    print(f"💼 Visiting job: {job_url}")
    driver.get(job_url)
    time.sleep(4)

    try:
        # Support both 'Apply' and 'Apply on company site'
        apply_button = None
        try:
            apply_button = driver.find_element(By.ID, "apply-button")
        except Exception:
            try:
                apply_button = driver.find_element(By.ID, "company-site-button")
            except Exception:
                pass

        if not apply_button:
            print("⚠️ No apply button found, skipping.")
            return

        btn_text = apply_button.text.strip().lower()
        print(f"🧭 Button text: {btn_text}")

        if "company site" in btn_text:
            print("🔗 Detected external job, saving.")
            save_external_job(job_url)
            return

        if "apply" in btn_text:
            driver.execute_script("arguments[0].scrollIntoView(true);", apply_button)
            driver.execute_script("arguments[0].click();", apply_button)
            print("✅ Clicked Apply.")
            time.sleep(4)
            # Give the modal / chat a short time to appear
            time.sleep(1.2)
            # If chatbot exists in DOM, handle it
            try:
                if driver.find_elements(By.CSS_SELECTOR, ".chatbot_MessageContainer") or driver.find_elements(By.CSS_SELECTOR, ".chatContainer") or driver.find_elements(By.CSS_SELECTOR, ".chat_msg_container"):
                    handle_chatbot_questions(driver, wait, resume_json)
                else:
                    print("ℹ️ No chatbot, single-click apply.")
            except Exception as e:
                print(f"⚠️ Error detecting chatbot: {e}")
        else:
            print("⚙️ Button unrecognized, skipping.")
    except Exception as e:
        print(f"❌ Error applying: {e}")

# ==============================
# MULTI-JOB HANDLER
# ==============================
def process_jobs_from_csv(csv_path, driver, wait, resume_json):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    df = pd.read_csv(csv_path)
    if "links" not in df.columns:
        raise ValueError("CSV must have a 'links' column.")

    print(f"📋 {len(df)} jobs found in CSV.")
    for idx, row in df.iterrows():
        job_url = str(row["links"]).strip()
        if not job_url.startswith("http"):
            continue
        print(f"\n{'='*80}\n🔗 [{idx+1}/{len(df)}] {job_url}")
        try:
            # NOTE: Resume upload is done once after login (outside this loop).
            apply_on_naukri(driver, wait, job_url, resume_json)
        except Exception as e:
            print(f"⚠️ Job error: {e}")
        time.sleep(2)
    print("\n🎯 All jobs processed successfully.")

# ==============================
# LOGIN + MAIN ENTRY
# ==============================
def ensure_logged_in(headless=False):
    email = os.getenv("NAUKRI_EMAIL")
    password = os.getenv("NAUKRI_PASSWORD")
    if not email or not password:
        raise RuntimeError("Set NAUKRI_EMAIL and NAUKRI_PASSWORD env vars.")

    driver = setup_driver(headless)
    wait = WebDriverWait(driver, 15)

    if os.path.exists(COOKIE_FILE):
        try:
            load_cookies(driver)
            if is_logged_in(driver):
                print("✅ Logged in via cookies.")
                return driver, wait
        except Exception as e:
            print(f"⚠️ Cookie login failed: {e}")

    driver.get(NAUKRI_HOMEPAGE)
    login_with_credentials(driver, wait, email, password)
    save_cookies(driver)
    return driver, wait

# ==============================
# MAIN EXECUTION
# ==============================
if __name__ == "__main__":
    try:
        with open(RESUME_JSON_PATH, "r", encoding="utf-8") as f:
            resume_json = json.load(f)
        print(f"📘 Loaded resume JSON from {RESUME_JSON_PATH}")

        driver, wait = ensure_logged_in(headless=False)

        # Upload resume once after login
        try:
            uploaded_ok = upload_resume_on_naukri(driver, wait)
            if not uploaded_ok:
                print("⚠️ Resume upload didn't succeed. Continuing but jobs may fail.")
        except Exception as e:
            print(f"⚠️ Resume upload raised exception: {e}")

        process_jobs_from_csv(JOBS_CSV_PATH, driver, wait, resume_json)

        driver.quit()
        print("✅ Done with all job applications.")
    except Exception as e:
        print(f"❌ Fatal error: {e}")


Gemini API key configured successfully ✅
📘 Loaded resume JSON from resumes/Yeswanth_Yerra_CV_structured.json
🍪 Cookies loaded and refreshed page.
✅ Logged in via cookies.
🚀 Uploading resume...
📄 Resume downloaded to /tmp/resume-drive.pdf
📤 Uploaded resume file input.
✅ Resume upload flow done.
📋 140 jobs found in CSV.

🔗 [1/140] https://www.naukri.com/job-listings-python-software-developer-pwc-india-bengaluru-7-to-11-years-111125041796
💼 Visiting job: https://www.naukri.com/job-listings-python-software-developer-pwc-india-bengaluru-7-to-11-years-111125041796
⚠️ No apply button found, skipping.

🔗 [2/140] https://www.naukri.com/job-listings-python-developer-pwc-india-bengaluru-6-to-11-years-121125035194
💼 Visiting job: https://www.naukri.com/job-listings-python-developer-pwc-india-bengaluru-6-to-11-years-121125035194
🧭 Button text: apply
✅ Clicked Apply.
🤖 Chatbot detected, starting Smart Q&A flow...
💬 Question: If you are not from Bangalore , are you willing to relocate to Bangalore


E0000 00:00:1763017564.812980    5399 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


🧠 Answer: Yes
☑️ Checked: Yes
💾 Clicked active Save button.
⏳ Waiting for next message or final message...
💬 Next question detected!
💬 Question: Overall experience and relevant experience
🧠 Answer: 0
💾 Clicked active Save button.
📨 Sent text answer (or attempted send).
⏳ Waiting for next message or final message...
🎯 Final thank-you message detected!
🏁 Application fully submitted ✅ Moving to next job.

🔗 [3/140] https://www.naukri.com/job-listings-python-developer-altimetrik-hyderabad-gurugram-5-to-8-years-111125020399
💼 Visiting job: https://www.naukri.com/job-listings-python-developer-altimetrik-hyderabad-gurugram-5-to-8-years-111125020399
⚠️ No apply button found, skipping.

🔗 [4/140] https://www.naukri.com/job-listings-python-software-developer-capgemini-bengaluru-6-to-11-years-111125013560
💼 Visiting job: https://www.naukri.com/job-listings-python-software-developer-capgemini-bengaluru-6-to-11-years-111125013560
🧭 Button text: apply
✅ Clicked Apply.
🤖 Chatbot detected, starting Sm

KeyboardInterrupt: 